# 🚀 Level 2 — Landmark, Alignment & Face Swap Klasik (OpenCV)

Notebook kedua dari roadmap **Learn-faceswap**. Di sini kita naik tingkat: bukan cuma *mendeteksi* wajah, tapi benar-benar **menukar wajah** memakai metode **klasik (tanpa deep learning)**.

Kenapa metode klasik dulu? Supaya kamu paham **mekanika** di balik face swap: titik landmark -> segitiga -> warp -> blending. Setelah paham ini, deep-learning (Roop/FaceFusion) akan terasa masuk akal.

**Alur notebook:**
1. Deteksi 468 titik **landmark** (MediaPipe Face Mesh)
2. **Alignment** — meluruskan wajah berdasarkan posisi mata
3. **Face Swap klasik** — Convex Hull -> Delaunay Triangulation -> Warp -> `seamlessClone`
4. Penjelasan hasil & perbandingan dengan metode AI

> 💡 Buka di **Google Colab**. Jalankan tiap cell dari atas (Shift+Enter). Tidak butuh GPU.

---

## 1️⃣ Install & Import

In [ ]:
!pip install -q opencv-python mediapipe matplotlib

import cv2
import numpy as np
import mediapipe as mp
import urllib.request
from matplotlib import pyplot as plt

mp_face_mesh = mp.solutions.face_mesh

def show(img, title='', size=(7, 7)):
    plt.figure(figsize=size)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis('off'); plt.show()

print('OpenCV:', cv2.__version__, '| MediaPipe:', mp.__version__)

## 2️⃣ Siapkan 2 gambar: SUMBER (wajah) + TARGET (badan/scene)

- **img_src** = gambar wajah yang ingin dipasang (yang diambil wajahnya)
- **img_dst** = gambar tujuan (wajahnya akan diganti dengan wajah dari img_src)

Default: download 2 contoh otomatis. Untuk pakai foto sendiri, aktifkan **Opsi B** (disarankan pakai fotomu sendiri — kamu yang punya hak atas gambarnya).

In [ ]:
# === Opsi A: contoh otomatis (2 wajah contoh dari repo tutorial publik) ===
src_url = 'https://raw.githubusercontent.com/ageitgey/face_recognition/master/examples/lin_manuel_miranda.jpg'
dst_url = 'https://raw.githubusercontent.com/ageitgey/face_recognition/master/examples/alex_lacamoire.jpg'
urllib.request.urlretrieve(src_url, 'src.jpg')
urllib.request.urlretrieve(dst_url, 'dst.jpg')
src_path, dst_path = 'src.jpg', 'dst.jpg'

# === Opsi B: upload foto sendiri (hapus tanda # untuk pakai) ===
# from google.colab import files
# print('Upload gambar SUMBER (wajah):')
# up1 = files.upload(); src_path = list(up1.keys())[0]
# print('Upload gambar TARGET:')
# up2 = files.upload(); dst_path = list(up2.keys())[0]

img_src = cv2.imread(src_path)
img_dst = cv2.imread(dst_path)

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(cv2.cvtColor(img_src, cv2.COLOR_BGR2RGB)); ax[0].set_title('SUMBER (wajah)'); ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(img_dst, cv2.COLOR_BGR2RGB)); ax[1].set_title('TARGET (akan diganti)'); ax[1].axis('off')
plt.show()

## 3️⃣ Deteksi Landmark (468 titik) + fungsi pembantu

`get_landmarks()` mengubah wajah menjadi 468 koordinat piksel. Inilah "peta wajah" yang jadi dasar segalanya.

In [ ]:
def get_landmarks(img):
    """Kembalikan array (N,2) koordinat piksel landmark, atau None jika tak ada wajah."""
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    with mp_face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1,
                              refine_landmarks=True, min_detection_confidence=0.5) as fm:
        res = fm.process(rgb)
        if not res.multi_face_landmarks:
            return None
        pts = [(int(lm.x * w), int(lm.y * h)) for lm in res.multi_face_landmarks[0].landmark]
        return np.array(pts, dtype=np.int32)

# Visualisasi landmark di kedua gambar
for name, im in [('SUMBER', img_src), ('TARGET', img_dst)]:
    pts = get_landmarks(im)
    vis = im.copy()
    if pts is not None:
        for (x, y) in pts:
            cv2.circle(vis, (x, y), 1, (0, 255, 0), -1)
        print(name, '-> jumlah titik:', len(pts))
    else:
        print(name, '-> WAJAH TIDAK TERDETEKSI!')
    show(vis, f'Landmark {name}', size=(6, 6))

## 4️⃣ Face Alignment (meluruskan wajah)

Sebelum menukar, wajah sering perlu **diluruskan** supaya mata sejajar horizontal. Caranya: hitung sudut kemiringan dari garis antar-mata, lalu putar gambar.

Index landmark MediaPipe: mata kanan ~ 33, mata kiri ~ 263.

In [ ]:
def align_face(img):
    pts = get_landmarks(img)
    if pts is None:
        return img
    right_eye, left_eye = pts[33], pts[263]
    dx, dy = left_eye[0] - right_eye[0], left_eye[1] - right_eye[1]
    angle = np.degrees(np.arctan2(dy, dx))
    h, w = img.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REFLECT_101)

aligned = align_face(img_src)
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(cv2.cvtColor(img_src, cv2.COLOR_BGR2RGB)); ax[0].set_title('Sebelum align'); ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB)); ax[1].set_title('Sesudah align (mata sejajar)'); ax[1].axis('off')
plt.show()

## 5️⃣ Mesin Face Swap Klasik

Tiga fungsi inti:
- **`calculate_delaunay_triangles`** — pecah wajah jadi banyak segitiga kecil (Delaunay triangulation)
- **`warp_triangle`** — tarik/regangkan tiap segitiga wajah sumber agar pas ke posisi wajah target
- (di langkah akhir) **`cv2.seamlessClone`** — tempel mulus + samakan warna kulit/cahaya

In [ ]:
def rect_contains(rect, point):
    return rect[0] <= point[0] <= rect[2] and rect[1] <= point[1] <= rect[3]

def calculate_delaunay_triangles(rect, points):
    subdiv = cv2.Subdiv2D(rect)
    for p in points:
        subdiv.insert((int(p[0]), int(p[1])))
    triangle_list = subdiv.getTriangleList()
    delaunay = []
    for t in triangle_list:
        tri = [(t[0], t[1]), (t[2], t[3]), (t[4], t[5])]
        if all(rect_contains(rect, pt) for pt in tri):
            idx = []
            for j in range(3):
                for k in range(len(points)):
                    if abs(tri[j][0] - points[k][0]) < 1.0 and abs(tri[j][1] - points[k][1]) < 1.0:
                        idx.append(k); break
            if len(idx) == 3:
                delaunay.append((idx[0], idx[1], idx[2]))
    return delaunay

def apply_affine_transform(src, src_tri, dst_tri, size):
    M = cv2.getAffineTransform(np.float32(src_tri), np.float32(dst_tri))
    return cv2.warpAffine(src, M, (size[0], size[1]), None,
                          flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)

def warp_triangle(img1, img2, t1, t2):
    r1 = cv2.boundingRect(np.float32([t1]))
    r2 = cv2.boundingRect(np.float32([t2]))
    t1_rect = [(t1[i][0] - r1[0], t1[i][1] - r1[1]) for i in range(3)]
    t2_rect = [(t2[i][0] - r2[0], t2[i][1] - r2[1]) for i in range(3)]
    mask = np.zeros((r2[3], r2[2], 3), dtype=np.float32)
    cv2.fillConvexPoly(mask, np.int32(t2_rect), (1.0, 1.0, 1.0), 16, 0)
    img1_rect = img1[r1[1]:r1[1] + r1[3], r1[0]:r1[0] + r1[2]]
    warped = apply_affine_transform(img1_rect, t1_rect, t2_rect, (r2[2], r2[3]))
    warped = warped * mask
    region = img2[r2[1]:r2[1] + r2[3], r2[0]:r2[0] + r2[2]].astype(np.float32)
    blended = region * ((1.0, 1.0, 1.0) - mask) + warped
    img2[r2[1]:r2[1] + r2[3], r2[0]:r2[0] + r2[2]] = blended.astype(img2.dtype)

print('Fungsi mesin face swap siap.')

In [ ]:
def face_swap(img_src, img_dst):
    """Tempel wajah dari img_src ke wajah di img_dst."""
    points1 = get_landmarks(img_src)
    points2 = get_landmarks(img_dst)
    if points1 is None or points2 is None:
        raise ValueError('Wajah tidak terdeteksi di salah satu gambar.')

    img_warped = np.copy(img_dst)

    # Convex hull = batas terluar wajah
    hull_index = cv2.convexHull(np.array(points2), returnPoints=False)
    hull1 = [points1[int(i)] for i in hull_index]
    hull2 = [points2[int(i)] for i in hull_index]

    # Delaunay triangulation di wajah target
    h, w = img_dst.shape[:2]
    triangles = calculate_delaunay_triangles((0, 0, w, h), hull2)
    if len(triangles) == 0:
        raise ValueError('Gagal membuat triangulation.')

    # Warp tiap segitiga dari sumber ke target
    for a, b, c in triangles:
        t1 = [hull1[a], hull1[b], hull1[c]]
        t2 = [hull2[a], hull2[b], hull2[c]]
        warp_triangle(img_src, img_warped, t1, t2)

    # Mask + seamless clone (penyatuan warna & tepi)
    mask = np.zeros(img_dst.shape, dtype=img_dst.dtype)
    cv2.fillConvexPoly(mask, np.int32([(p[0], p[1]) for p in hull2]), (255, 255, 255))
    r = cv2.boundingRect(np.float32([hull2]))
    center = (r[0] + r[2] // 2, r[1] + r[3] // 2)
    return cv2.seamlessClone(np.uint8(img_warped), img_dst, mask, center, cv2.NORMAL_CLONE)

result = face_swap(img_src, img_dst)
show(result, 'HASIL FACE SWAP (wajah SUMBER di badan TARGET)', size=(7, 7))

## 6️⃣ Bandingkan hasil

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 6))
for a, im, t in zip(ax, [img_src, img_dst, result], ['SUMBER', 'TARGET asli', 'HASIL SWAP']):
    a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.set_title(t); a.axis('off')
plt.show()

## 7️⃣ Penjelasan Hasil & Apa yang Bisa Diharapkan

**Cara kerja yang baru saja terjadi:**
1. Wajah dipetakan jadi 468 titik (landmark)
2. Batas wajah (convex hull) dibagi jadi puluhan segitiga (Delaunay)
3. Tiap segitiga wajah sumber **diregangkan** agar pas ke posisi segitiga wajah target (warp)
4. `seamlessClone` menyamarkan tepi & menyamakan warna kulit/cahaya

**Yang realistis diharapkan dari metode KLASIK ini:**
- ✅ Cepat, jalan di CPU, gratis, mudah dipahami
- ✅ Bagus jika kedua wajah pose-nya mirip (sama-sama menghadap depan)
- ⚠️ Kurang rapi jika beda pose/pencahayaan ekstrem; tekstur kulit bisa terlihat "tertempel"
- ❌ Tidak bisa membuat ekspresi/sudut baru — hanya menempel & meregang

**Inilah kenapa ada metode DEEP LEARNING** (Roop/FaceFusion/DeepFaceLab) yang akan kita pakai di Level 4: ia benar-benar *meng-generate* ulang wajah agar menyatu natural.

### 🧠 Latihan
- Coba pakai 2 fotomu sendiri (Opsi B di Cell 2)
- Tukar urutan: jadikan TARGET sebagai SUMBER dan sebaliknya
- Ganti `cv2.NORMAL_CLONE` menjadi `cv2.MIXED_CLONE`, amati bedanya

### 👉 Berikutnya (Level 3 & 4)
- **Notebook 3:** Face swap pakai **deep learning** (model `inswapper` / InsightFace) — hasil jauh lebih natural
- **Notebook 4:** Pakai tool jadi **FaceFusion/Roop** untuk swap di video